# TL-Bot — char_classifier Training

**Before the very first run:** set runtime to GPU — *Runtime → Change runtime type → T4 GPU → Save*

**Every session: click *Runtime → Run all*.** That's it.
- Cell 1 mounts Drive.
- Cell 2 clones or pulls the repo, then starts training. If a checkpoint exists on Drive it resumes automatically; otherwise it starts fresh.

Checkpoints are saved to Drive after every epoch. When the runtime is recycled, open a new session and run all again.

---
**One-time prerequisite (local, before the first Colab session):**
```powershell
# From the repo root — zip the latin dataset and upload to Drive
.venv\Scripts\python.exe Models\colab_train.py --zip-dataset --scripts latin
# Then upload char-dataset.zip to: My Drive/Colab Notebooks/TL-Bot/
```

In [ ]:
import os, torch
from pathlib import Path

CKPT = "/content/drive/MyDrive/Colab Notebooks/TL-Bot/checkpoints/latin"
for name in ("last.pt", "best.pt"):
    p = Path(CKPT) / name
    if p.exists():
        ckpt = torch.load(str(p), map_location="cpu", weights_only=False)
        mtime = p.stat().st_mtime
        import datetime
        t = datetime.datetime.fromtimestamp(mtime).strftime("%Y-%m-%d %H:%M:%S")
        print(f"{name}: epoch={ckpt.get('epoch','?')}  val_acc={ckpt.get('best_val_acc', ckpt.get('val_acc','?')):.4f}  saved={t}")
    else:
        print(f"{name}: not found")


---
## Cell 1 — Mount Drive
Required every session. Auth popup appears on first mount.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## Cell 2 — Train
Run every session. Clones the repo on first run, pulls updates on subsequent runs.
Resumes from the latest Drive checkpoint automatically; starts fresh if none exists.

In [ ]:
import os, subprocess

REPO_URL  = "https://github.com/alexjade96/Discord-TL_Bot.git"
REPO_DIR  = "/content/Discord-TL_Bot"

# Clone on first session; pull updates on subsequent sessions.
if os.path.isdir(f"{REPO_DIR}/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

# --resume is safe to pass every time: colab_train.py starts fresh
# automatically when no checkpoint is found on Drive.
subprocess.run(
    ["python", f"{REPO_DIR}/Models/colab_train.py", "--skip-clone", "--resume"],
    check=True,
)

---
## Cell 3 — Smoke Test *(optional — do not include in Run All)*
Quick sanity check before committing to a full run: 10 images/class, 4 epochs.
Run this cell manually only; do not use *Run All* when this cell is present.

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/alexjade96/Discord-TL_Bot.git"
REPO_DIR = "/content/Discord-TL_Bot"

if os.path.isdir(f"{REPO_DIR}/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

subprocess.run(
    ["python", f"{REPO_DIR}/Models/colab_train.py", "--skip-clone", "--smoke-test"],
    check=True,
)